# 1. Introduction Database

## 1.1 Apa itu database

Database: kumpulan data yang disimpan secara terstruktur sehingga bisa diakses, dikelola dan diperbarui secara sistematis.

Database dirancang agar:
* Data konsisten (tidak ada versi ganda yang saling bertentangan)
* Bisa diakses banyak pengguna/aplikasi secara bersamaan
* Bisa di-query dengan cepat meski volumenya besar

## 1.2 Relational Database

Relational Database (RDBMS): menyimpan data dalam bentuk tabel tabel yang saling terhubung melalui key. Ciri khas:
*  Skema (struktur tabel dan tipe data) bersifat ketat dan didefinisikan di awal
* Data dipecah ke banyak tabel kecil yang saling berelasi, bukan satu tabel raksasa
* Berbeda dengan NoSQL (dokumen/key-value) yang jauh lebih fleksibel tapi kurang ketat strukturnya 

## 1.3 Database vs Table, Row vs Column

Database adalah wadah besar yang berisi banyak table. Table sendiri berbentuk grid mirip spreadsheet, tapi dengan aturan tipe data yang ketat per kolom.

* Row = satu record/observasi (misal: satu customer, satu transaksi)
* Column = satu atribut dari record itu (misal: nama, email, tanggal registrasi)

persis seperti DataFrame di pandas — row = index/baris data, column = fitur/variabel.

## 1.4 Primary Key dan Foreign Key

* Primary Key (PK): kolom yang secara unik mengidentifikasi setiap row dalam satu tabel. Tidak boleh duplikat, tidak boleh NULL. Contoh: customer_id di tabel customers.
* Foreign Key (FK): kolom di suatu tabel yang menunjuk ke Primary Key di tabel lain, untuk membangun relasi antar tabel. Contoh: customer_id di tabel orders adalah FK yang merujuk ke PK customer_id di tabel customers.

Dengan FK ini, sistem tahu "order ini milik customer siapa" tanpa harus menyimpan ulang seluruh data customer di setiap baris order — inilah alasan data dipecah ke banyak tabel, bukan digabung jadi satu.

## 1.5 Schema, SQL, dan DBMS

* Schema: blueprint database — mendefinisikan tabel apa saja yang ada, kolom dan tipe datanya, serta relasi antar tabel.
* SQL (Structured Query Language): bahasa standar untuk berkomunikasi dengan relational database — mengambil, menyaring, mengubah, atau menghapus data.
* DBMS (Database Management System): software yang mengelola database itu sendiri (menyimpan data, menjalankan query, menjaga konsistensi). PostgreSQL adalah contoh DBMS; SQL adalah bahasa yang Anda gunakan untuk "berbicara" dengannya.

![database_structure_ecommerce.png](../assets/database_structure_ecommerce.png)

Komponen-komponen utama:

* Tabel customers (biru): menyimpan data customer. Kolom customer_id adalah Primary Key (ditandai 🔑) — setiap customer punya ID unik yang tidak boleh duplikat.
* Tabel categories (hijau): menyimpan kategori produk. category_id adalah PK-nya.
* Tabel products (kuning): menyimpan produk. product_id adalah PK-nya, tapi ada kolom lain bernama category_id yang ditandai 🔗 (Foreign Key) — ini menunjuk kembali ke tabel categories. Artinya: setiap produk harus punya kategori yang valid di tabel categories.
* Tabel orders (teal): menyimpan order dari customer. order_id adalah PK-nya, dan customer_id adalah FK yang menunjuk ke tabel customers — setiap order harus milik customer yang ada.

Relasi antar tabel:

Panah putus-putus menunjukkan relasi one-to-many:
* Satu customer bisa punya banyak order — tetapi satu order hanya milik satu customer.
* Satu kategori bisa punya banyak produk — tetapi satu produk hanya milik satu kategori.

# 2. Implementasi — Memetakan Struktur Database E-commerce

Database yang akan dipakai punya struktur seperti ini:

```text
Database: ds_sql_learning
│
├── customers   (customer_id [PK], name, email, city, ...)
├── categories  (category_id [PK], category_name)
├── products    (product_id [PK], name, price, category_id [FK → categories])
└── orders      (order_id [PK], customer_id [FK → customers], order_date, ...)

* customers dan orders berelasi one-to-many: satu customer bisa punya banyak order, tapi satu order hanya milik satu customer. Relasi ini terbentuk lewat customer_id (PK di customers, FK di orders).
* products dan categories juga one-to-many: satu kategori bisa punya banyak produk, lewat category_id (PK di categories, FK di products).
* Perhatikan: tabel orders tidak menyimpan nama atau email customer secara langsung — ia cukup menyimpan customer_id, lalu detail customer diambil dengan menghubungkan ke tabel customers saat dibutuhkan. Ini prinsip dasar relational design: hindari duplikasi data.

# 3. SELECT

## 3.1 Anatomi Dasar Query SELECT

Bentuk paling sederhana

```SQL
SELECT column_list
FROM table_name;

Beberapa hal fundamental
* Keyword tidak case sensitive (SELECT = select)
* Semicolon (;) menandai akhir statement
* Urutan eksekusi SQL: `FROM` → `SELECT`.
    * `FROM` menentukan tabel sumber, lalu `SELECT` memilih/menghitung kolom.
    * Konsep ini penting untuk memahami `WHERE` dan `GROUP BY` nantinya.


## 3.2 SELECT * vs Kolom Eksplisit

```sql
SELECT * FROM customers;

" * " = semua kolom.

penggunaan * dihindari karena:
* Kalau schema tabel berubah (ada kolom baru ditambahkan), hasil query ikut berubah tanpa Anda sadari — ini bisa mematahkan pipeline/dashboard yang bergantung pada urutan atau jumlah kolom tertentu.
* Menarik kolom yang tidak dibutuhkan membebani transfer data, apalagi di tabel dengan kolom besar (misal kolom teks panjang atau JSON).
* Kolom eksplisit membuat intent query jelas dibaca orang lain (atau diri Anda sendiri enam bulan kemudian).

Bandingkan dengan versi eksplisit

```sql

SELECT
    customer_id,
    name,
    email
FROM customers;
```

Urutan kolom di hasil mengikuti urutan yang Anda tulis di SELECT, bukan urutan asli di tabel.

## 3.3 Alias Kolom (AS)

Alias mengganti nama tampilan kolom di hasil query, tanpa mengubah nama asli di database:

```sql
SELECT 
    name AS customers_name,
    email AS contact_email,
FROM customers;
```

* AS > opsional (name customer_name juga valid), ditulis untuk keterbacaan
* Kalau alias mengandung spasi atau kata yang bentrok dengan reserved word, PostgreSQL butuh tanda kutip ganda: SELECT price AS "Price (IDR)"
* Alias berguna terutama nanti saat hasil ekspresi/agregasi butuh nama yang lebih deskriptif daripada sum, count, dsb

## 3.4 Expression dalam SELECT

SELECT tidak cuma menarik kolom mentah, tapi bisa menghitung kolom baru on the fly, mirip df.assign() di pandas:

```sql
SELECT 
    name,
    price * 1.11 AS price_with_tax
FROM product;

Atau menggabungkan string (PostgreSQL pakai ||, bukan + seperti sebagian bahasa lain):

```sql
SELECT
    name || ' - ' || email AS contact_info
FROM customers;

## 3.5 Komentar

```sql
-- komentar satu baris
SELECT * FROM customers;

/* komentar
   multi baris */